# Task 4 — Statistical modeling & risk-based pricing (ACIS)

**Goals**

1. **Claim severity** — on rows with `TotalClaims > 0`, predict `TotalClaims` (RMSE, R²).
2. **Claim incidence** — classify `has_claim` at row level (accuracy, precision, recall, F1).
3. **Pricing skeleton** — naive benchmark `CalculatedPremiumPerTerm` vs `risk_adjusted_premium` using `P(claim)` × predicted severity + loadings.

Reusable code: `src/modeling.py`. **SHAP** bar plot for the best tree model is written under `reports/figures/` when you run the last cells.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_insurance_data
from src.modeling import (
    ModelTrainer,
    engineer_features,
    naive_premium_baseline,
    risk_adjusted_premium,
    save_shap_summary,
    select_feature_columns,
)

# Use None for full data; use e.g. 200_000 for faster iteration
NROWS = None

df = load_insurance_data(nrows=NROWS)
print(df.shape)
df.head(2)

## Severity models (regression)

Algorithms: **linear regression**, **random forest**, **XGBoost**.

In [ ]:
trainer = ModelTrainer(random_state=42)
severity_metrics, severity_models = trainer.fit_eval_severity(df, test_size=0.2)
severity_metrics.sort_values("rmse")

## Claim classifier

Linear baseline: **logistic regression**; ensembles: **random forest**, **XGBoost**.

In [ ]:
cls_metrics, cls_models = trainer.fit_eval_claim_classifier(df, test_size=0.2)
cls_metrics

## Pricing illustration

For a rough technical premium on **all rows** (not production-ready), combine predicted `P(claim)` with **expected severity** from the severity model trained on positive claims only. Here we blend with the **portfolio mean severity** for rows without a severity prediction (cold start).

In [ ]:
import numpy as np

d_eng = engineer_features(df)
num, cat = select_feature_columns(d_eng)
X_all = d_eng[num + cat]

best_cls_name = cls_metrics.sort_values("f1", ascending=False).iloc[0]["model"]
clf_pipe = cls_models[str(best_cls_name)]
p_claim = clf_pipe.predict_proba(X_all)[:, 1]

best_sev_name = severity_metrics.sort_values("rmse").iloc[0]["model"]
sev_pipe = severity_models[str(best_sev_name)]
pos_mask = pd.to_numeric(d_eng["TotalClaims"], errors="coerce").fillna(0) > 0
mean_sev = float(pd.to_numeric(d_eng.loc[pos_mask, "TotalClaims"], errors="coerce").mean())
sev_hat = np.full(len(d_eng), mean_sev)
sev_hat[pos_mask.to_numpy()] = sev_pipe.predict(X_all.loc[pos_mask])

tech_prem = risk_adjusted_premium(p_claim, sev_hat)
naive = naive_premium_baseline(d_eng)
pd.DataFrame(
    {
        "naive_calculated": naive[:5],
        "technical_risk_based": tech_prem[:5],
    }
)

## SHAP interpretability (best severity tree model)

We take a random subsample for speed and plot **mean |SHAP|** for the top features. In business language, positive SHAP on `vehicle_age_years` means older vehicles push the predicted claim amount **up** relative to the model baseline, holding other encoded features fixed.

In [ ]:
fig_dir = ROOT / "reports" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

d_pos = d_eng[pos_mask].copy()
X_pos = d_pos[num + cat]
sample = X_pos.sample(min(2000, len(X_pos)), random_state=42)

cand = severity_metrics[severity_metrics["model"].isin(["random_forest", "xgboost"])].sort_values("rmse")
if cand.empty:
    raise RuntimeError("Need at least one tree model row for SHAP.")
best_tree = str(cand.iloc[0]["model"])
out = save_shap_summary(
    severity_models[best_tree],
    sample,
    fig_dir / "shap_severity_best.png",
    max_display=10,
)
print("Saved:", out)